# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library and its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display metadata summary
meta = dataset.metadata  # NOTE: Don't subscript; use attributes.
print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.date_published}")
print(f"Version: {meta.version}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s. We'll inspect all record sets and their schema from the metadata.

In [ ]:
# List out all record sets in the dataset's metadata

print('Record Sets:')
record_sets = dataset.metadata.record_sets
record_set_ids = []

for rs in record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', '')}")
    record_set_ids.append(rs.id)
    # List fields in each record set
    fields = rs.fields
    print("  Fields:")
    field_ids = []
    for fld in fields:
        print(f"    - {fld.name if hasattr(fld,'name') else ''} (@id: {fld.id}) [type={getattr(fld, 'data_type', '')}]")
        field_ids.append(fld.id)
    print()

# Choose the first record set as a default for demonstration
if record_set_ids:
    default_record_set_id = record_set_ids[0]
    print(f"We'll use record set: {default_record_set_id}\n")
else:
    raise ValueError("No record sets found in dataset.")


## 3. Data Extraction
Load data from chosen record set(s) into pandas DataFrames. Use `@id`s for all entities.

> We'll load each record set and view available columns for further analysis.

In [ ]:
# Extract data from each record set into DataFrames
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    recs = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(recs)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}")
    print()

# Pick the default record set for EDA/visualization
df_main = dataframes[default_record_set_id]
print("Sample records:")
display(df_main.head())

## 4. Exploratory Data Analysis (EDA)
Let's apply some initial data processing steps:
- Pick a numeric field for filtering/normalization
- Filter records based on a threshold
- Normalize the field
- Optionally group by a key attribute

All fields are referenced using their `@id` from step 2.

In [ ]:
# Identify numeric fields in the default record set for analysis
rs = None
for r in dataset.metadata.record_sets:
    if r.id == default_record_set_id:
        rs = r
        break
if rs is None:
    raise ValueError("Default record set id could not be located.")

# Pick numeric fields (e.g. MW, Coverage, PeptideCount, etc. -- check actual field names/columns)
numeric_fields = []
for fld in rs.fields:
    if getattr(fld, 'data_type', None) in ('Float', 'Integer', 'Number'):
        numeric_fields.append(fld.id)

if not numeric_fields:
    raise ValueError("No numeric fields found for EDA.")
# Pick the first numeric field for demonstration
numeric_field = numeric_fields[0]
print(f"Using numeric field for filtering: {numeric_field}")

# Filter: keep only rows with numeric_field > threshold
threshold = df_main[numeric_field].quantile(0.75) if pd.api.types.is_numeric_dtype(df_main[numeric_field]) else 10.0

filtered_df = df_main[df_main[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, norm_col]].head())

# Optional grouping: use first non-numeric field
group_field = None
for fld in rs.fields:
    if fld.id not in numeric_fields:
        group_field = fld.id
        break

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped by {group_field}, mean of {numeric_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field or its relationship with a categorical/grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df_main[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If a group field is available, plot group means
if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(10, 4))
    sns.barplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.xticks(rotation=90)
    plt.title(f'{numeric_field} by {group_field} (filtered)')
    plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded metadata and records from the FAIR^2 dataset using the Croissant schema and `mlcroissant`.
- Explored all available record sets and fields using their `@id` identifiers.
- Loaded records into Pandas DataFrames and performed basic EDA on key numeric attributes, including filtering, normalization, and optional grouping.
- Visualized field distributions to further understand the dataset's structure and patterns.

Go further by customizing threshold values, filtering using other field `@id`s, or extending to advanced analyses!